In [2]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown
import ipywidgets as widgets
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D  # for 3D plotting

%matplotlib inline
np.random.seed(42)

In [3]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_deriv(a):
    
    return a * (1 - a)

def tanh(x):
    return np.tanh(x)

def tanh_deriv(a):
    
    return 1 - a**2


In [6]:
# Generate synthetic data

N = 20  # number of samples
X_data = np.linspace(-1, 1, N).reshape(-1, 1)  # shape (N,1)
Y_reg = np.sin(np.pi * X_data)                  # regression target
Y_class = (Y_reg >= 0).astype(np.float32)         # binary classification target


In [8]:
X_data

array([[-1.        ],
       [-0.89473684],
       [-0.78947368],
       [-0.68421053],
       [-0.57894737],
       [-0.47368421],
       [-0.36842105],
       [-0.26315789],
       [-0.15789474],
       [-0.05263158],
       [ 0.05263158],
       [ 0.15789474],
       [ 0.26315789],
       [ 0.36842105],
       [ 0.47368421],
       [ 0.57894737],
       [ 0.68421053],
       [ 0.78947368],
       [ 0.89473684],
       [ 1.        ]])

In [8]:
def train_network(problem_type='classification', optimizer='sgd', epochs=50, lr=0.01):
    # Network dimensions:
    in_dim = 1
    h1_dim = 2
    h2_dim = 3
    out_dim = 1
    
    # Initialize weights and biases with small random values.
    W1 = np.random.randn(in_dim, h1_dim) * 0.1
    b1 = np.zeros((1, h1_dim))
    W2 = np.random.randn(h1_dim, h2_dim) * 0.1
    b2 = np.zeros((1, h2_dim))
    W3 = np.random.randn(h2_dim, out_dim) * 0.1
    b3 = np.zeros((1, out_dim))
    
    # Group parameters in a dictionary.
    params = {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2, 'W3': W3, 'b3': b3}
    
    # Set up state for optimizers that require extra storage.
    state = {}
    if optimizer == 'momentum':
        for key in params:
            state[key] = np.zeros_like(params[key])
    elif optimizer == 'rmsprop':
        for key in params:
            state[key] = np.zeros_like(params[key])
    elif optimizer == 'adam':
        for key in params:
            state["m_" + key] = np.zeros_like(params[key])
            state["v_" + key] = np.zeros_like(params[key])
    elif optimizer == 'adagrad':
        for key in params:
            state[key] = np.zeros_like(params[key])
    
    # Prepare dictionaries to store trajectories for each scalar parameter and the loss.
    traj = {}
    for key in params:
        shape = params[key].shape
        for index in np.ndindex(shape):     #for a (2, 3) array, it would iterate over indices (0, 0), (0, 1), (0, 2), (1, 0), (1, 1), and (1, 2)
            traj[f"{key}{index}"] = []  
    loss_history = []
    
    n_samples = X_data.shape[0]
    t = 0  # iteration counter (for Adam bias correction)
    
    # Training loop (SGD style: one sample per iteration)
    for epoch in range(epochs):
        indices = np.random.permutation(n_samples)
        for i in indices:
            t += 1
            # Get one sample
            x = X_data[i].reshape(1, in_dim)
            y = Y_class[i].reshape(1, out_dim) if problem_type == 'classification' else Y_reg[i].reshape(1, out_dim)
            
            #  Forward pass...
            # Hidden Layer 1
            z1 = np.dot(x, params['W1']) + params['b1']    # shape (1,2)
            a1 = tanh(z1)                                  # shape (1,2)
            # Hidden Layer 2
            z2 = np.dot(a1, params['W2']) + params['b2']     # shape (1,3)
            a2 = tanh(z2)                                  # shape (1,3)
            # Output Layer
            z3 = np.dot(a2, params['W3']) + params['b3']     # shape (1,1)
            if problem_type == 'classification':
                a3 = sigmoid(z3)
                loss = - ( y * np.log(a3 + 1e-8) + (1 - y) * np.log(1 - a3 + 1e-8) )
            else:
                a3 = z3  # linear activation for regression
                loss = 0.5 * (a3 - y)**2
            loss = loss.flatten()[0]
            loss_history.append(loss)
            
            #  Backpropagation...
            d3 = a3 - y  # works for both loss functions in this setup (shape: (1,1))
            dW3 = np.dot(a2.T, d3)  # shape (3,1)
            db3 = d3              # shape (1,1)
            
            da2 = np.dot(d3, params['W3'].T)  # shape (1,3)
            dz2 = da2 * tanh_deriv(a2)        # shape (1,3)
            dW2 = np.dot(a1.T, dz2)           # shape (2,3)
            db2 = dz2                       # shape (1,3)
            
            da1 = np.dot(dz2, params['W2'].T) # shape (1,2)
            dz1 = da1 * tanh_deriv(a1)        # shape (1,2)
            dW1 = np.dot(x.T, dz1)            # shape (1,2)
            db1 = dz1                       # shape (1,2)
            
            grads = {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2, 'W3': dW3, 'b3': db3}
            
            # Parameter update using optimizer....
            for key in params:
                grad = grads[key]
                if optimizer == 'sgd':
                    params[key] = params[key] - lr * grad
                elif optimizer == 'momentum':
                    state[key] = 0.9 * state[key] + lr * grad
                    params[key] = params[key] - state[key]
                elif optimizer == 'rmsprop':
                    state[key] = 0.9 * state[key] + (1 - 0.9) * (grad**2)
                    params[key] = params[key] - lr * grad / (np.sqrt(state[key]) + 1e-8)
                elif optimizer == 'adam':
                    state["m_" + key] = 0.9 * state["m_" + key] + (1 - 0.9) * grad
                    state["v_" + key] = 0.999 * state["v_" + key] + (1 - 0.999) * (grad**2)
                    m_hat = state["m_" + key] / (1 - 0.9**t)
                    v_hat = state["v_" + key] / (1 - 0.999**t)
                    params[key] = params[key] - lr * m_hat / (np.sqrt(v_hat) + 1e-8)
                elif optimizer == 'adagrad':
                    state[key] = state[key] + grad**2
                    params[key] = params[key] - lr * grad / (np.sqrt(state[key]) + 1e-8)
                else:
                    raise ValueError("Unknown optimizer.")
            
            # Record current parameters...
            for key in params:
                for index in np.ndindex(params[key].shape):
                    traj[f"{key}{index}"].append(params[key][index])
                    
    return traj, loss_history, params


In [10]:
optimizers = ['sgd', 'momentum', 'rmsprop', 'adam', 'adagrad']
results = {'classification': {}, 'regression': {}}
epochs = 50
lr = 0.01

for prob in ['classification', 'regression']:
    for opt in optimizers:
        print(f"Training {prob} problem using {opt} optimizer...")
        traj, loss_history, final_params = train_network(problem_type=prob, optimizer=opt, epochs=epochs, lr=lr)
        results[prob][opt] = {'traj': traj, 'loss': loss_history, 'final_params': final_params}
        print("Final loss: {:.5f}".format(loss_history[-1]))


Training classification problem using sgd optimizer...
Final loss: 0.69559
Training classification problem using momentum optimizer...
Final loss: 0.00486
Training classification problem using rmsprop optimizer...
Final loss: 0.00000
Training classification problem using adam optimizer...
Final loss: 0.00037
Training classification problem using adagrad optimizer...
Final loss: 0.24552
Training regression problem using sgd optimizer...
Final loss: 0.00001
Training regression problem using momentum optimizer...
Final loss: 0.04865
Training regression problem using rmsprop optimizer...
Final loss: 0.05732
Training regression problem using adam optimizer...
Final loss: 0.05798
Training regression problem using adagrad optimizer...
Final loss: 0.16503


In [28]:
# %% [code] 3D Visualization: Plot a 3D curve of (weight, bias, loss)
# x-axis: evolution of a specific weight
# y-axis: corresponding bias evolution
# z-axis: loss at each iteration
#
# The preset_pairs dictionary has been updated so the keys match the stored names (including spaces).
preset_pairs = {
    "Layer1: W1(0, 0) & b1(0, 0)": ("W1(0, 0)", "b1(0, 0)"),
    "Layer1: W1(0, 1) & b1(0, 1)": ("W1(0, 1)", "b1(0, 1)"),
    "Layer2: W2(0, 0) & b2(0, 0)": ("W2(0, 0)", "b2(0, 0)"),
    "Layer2: W2(0, 1) & b2(0, 1)": ("W2(0, 1)", "b2(0, 1)"),
    "Layer2: W2(0, 2) & b2(0, 2)": ("W2(0, 2)", "b2(0, 2)"),
    "Output: W3(0, 0) & b3(0, 0)": ("W3(0, 0)", "b3(0, 0)")
}

def plot_3d_curve_for_pair(pair_name, problem, optimizer):
    traj = results[problem][optimizer]['traj']
    loss_history = results[problem][optimizer]['loss']
    weight_key, bias_key = preset_pairs[pair_name]
    
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    x_vals = traj[weight_key]
    y_vals = traj[bias_key]
    z_vals = loss_history
    ax.plot(x_vals, y_vals, z_vals, marker='o', label=pair_name)
    ax.set_xlabel("Weight")
    ax.set_ylabel("Bias")
    ax.set_zlabel("Loss")
    ax.set_title(f"3D Evolution: {pair_name}")
    ax.legend()
    plt.show()

def interactive_3d_plot():
    problem_dropdown = Dropdown(options=['classification', 'regression'], description='Problem:')
    optimizer_dropdown = Dropdown(options=optimizers, description='Optimizer:')
    pair_dropdown = Dropdown(options=list(preset_pairs.keys()), description='Param Pair:')
    interact(lambda problem, optimizer, pair: plot_3d_curve_for_pair(pair, problem, optimizer),
             problem=problem_dropdown, optimizer=optimizer_dropdown, pair=pair_dropdown)

print("Interactive 3D visualization for parameter pair evolution (Weight, Bias, Loss):")
interactive_3d_plot()


Interactive 3D visualization for parameter pair evolution (Weight, Bias, Loss):


interactive(children=(Dropdown(description='Problem:', options=('classification', 'regression'), value='classi…

In [26]:
# %% [code] Interactive visualization: Plot parameter trajectory and loss curve.
# This interactive tool lets you choose a specific parameter (e.g., "W1(0,0)" or "b1(0,)") and view:
#   - Its value evolution (error surface) over iterations.
#   - The corresponding loss curve.

def plot_parameter(param_name, traj, loss_history):
    fig, axs = plt.subplots(2, 1, figsize=(8, 6))
    iterations = range(len(traj[param_name]))
    
    # Plot parameter value evolution
    axs[0].plot(iterations, traj[param_name], label=param_name, color='blue')
    axs[0].set_xlabel("Iteration")
    axs[0].set_ylabel("Parameter value")
    axs[0].set_title(f"Trajectory of {param_name}")
    axs[0].legend()
    
    # Plot loss curve
    axs[1].plot(iterations, loss_history, label="Loss", color='red')
    axs[1].set_xlabel("Iteration")
    axs[1].set_ylabel("Loss")
    axs[1].set_title("Loss Curve")
    axs[1].legend()
    
    plt.tight_layout()
    plt.show()

def interactive_plot(problem, optimizer):
    traj = results[problem][optimizer]['traj']
    loss_history = results[problem][optimizer]['loss']
    param_keys = list(traj.keys())
    param_keys.sort()
    dropdown = Dropdown(options=param_keys, description='Parameter:')
    interact(lambda param: plot_parameter(param, traj, loss_history), param=dropdown)

print("Interactive visualization for parameter trajectories and loss curves:")
print("Select a problem type and optimizer below:")
problem_dropdown = Dropdown(options=['classification', 'regression'], description='Problem:')
optimizer_dropdown = Dropdown(options=optimizers, description='Optimizer:')

def update_interactive(problem, optimizer):
    interactive_plot(problem, optimizer)

widgets.interact(update_interactive, problem=problem_dropdown, optimizer=optimizer_dropdown)


Interactive visualization for parameter trajectories and loss curves:
Select a problem type and optimizer below:


interactive(children=(Dropdown(description='Problem:', options=('classification', 'regression'), value='classi…

<function __main__.update_interactive(problem, optimizer)>

In [16]:

def display_param_table(param_name, traj):
    df = pd.DataFrame({
        "Iteration": list(range(len(traj[param_name]))),
        "Value": traj[param_name]
    })
    display(df)

def interactive_table(problem, optimizer):
    traj = results[problem][optimizer]['traj']
    param_keys = list(traj.keys())
    param_keys.sort()
    dropdown = Dropdown(options=param_keys, description='Parameter:')
    interact(lambda param: display_param_table(param, traj), param=dropdown)

print("Interactive parameter value table:")
print("Select a problem type and optimizer below:")
widgets.interact(interactive_table, problem=problem_dropdown, optimizer=optimizer_dropdown)


Interactive parameter value table:
Select a problem type and optimizer below:


interactive(children=(Dropdown(description='Problem:', options=('classification', 'regression'), value='classi…

<function __main__.interactive_table(problem, optimizer)>

In [18]:
# %% [code] (Optional) Display final parameters in a table.
def display_final_params(problem, optimizer):
    final_params = results[problem][optimizer]['final_params']
    rows = []
    for key in final_params:
        arr = final_params[key]
        for index in np.ndindex(arr.shape):
            rows.append({"Parameter": f"{key}{index}", "Value": arr[index]})
    df = pd.DataFrame(rows)
    display(df)

print("Final parameters for classification problem using Adam optimizer:")
display_final_params('classification', 'adam')


Final parameters for classification problem using Adam optimizer:


,Parameter,Value
0,"W1(0, 0)",2.477521
1,"W1(0, 1)",2.473016
2,"b1(0, 0)",-0.000549
3,"b1(0, 1)",-0.010163
4,"W2(0, 0)",-2.130506
5,"W2(0, 1)",2.088897
6,"W2(0, 2)",1.965438
7,"W2(1, 0)",-2.381132
8,"W2(1, 1)",2.475332
9,"W2(1, 2)",2.314879


In [73]:
# Observations based on the training curves:
#
# - SGD:
#     • Loss is relatively noisy.
#     • Weights and biases update gradually.
#
# - Momentum:
#     • Convergence is faster with visible momentum in the updates.
#
# - RMSprop:
#     • Smoother loss decrease; parameters adjust based on recent gradient history.
#
# - Adam:
#     • Rapid and consistent convergence with steady parameter evolution.
#
# - Adagrad:
#     • Fast initial progress with slower adjustments later.


In [30]:
# %% [code] Import necessary libraries and set seed
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown
import ipywidgets as widgets
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D  # for 3D plotting

%matplotlib inline
np.random.seed(42)


In [32]:
# %% [code] Define activation functions and their derivatives

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_deriv(a):
    return a * (1 - a)

def tanh(x):
    return np.tanh(x)

def tanh_deriv(a):
    return 1 - a**2


In [34]:


N = 20  # number of samples
X_data = np.linspace(-1, 1, N).reshape(-1, 1)  # shape (N,1)
Y_reg = np.sin(np.pi * X_data)                  # regression target
Y_class = (Y_reg >= 0).astype(np.float32)         # binary classification target


In [38]:


def train_network(problem_type='classification', optimizer='sgd', epochs=50, lr=0.01):
    # Network dimensions:
    in_dim = 1
    h1_dim = 2
    h2_dim = 3
    out_dim = 1
    
    # Initialize weights and biases with small random values.
    W1 = np.random.randn(in_dim, h1_dim) * 0.1
    b1 = np.zeros((1, h1_dim))
    W2 = np.random.randn(h1_dim, h2_dim) * 0.1
    b2 = np.zeros((1, h2_dim))
    W3 = np.random.randn(h2_dim, out_dim) * 0.1
    b3 = np.zeros((1, out_dim))
    
    # Group parameters in a dictionary.
    params = {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2, 'W3': W3, 'b3': b3}
    
    # Set up state for optimizers that require extra storage.
    state = {}
    if optimizer == 'momentum':
        for key in params:
            state[key] = np.zeros_like(params[key])
    elif optimizer == 'rmsprop':
        for key in params:
            state[key] = np.zeros_like(params[key])
    elif optimizer == 'adam':
        for key in params:
            state["m_" + key] = np.zeros_like(params[key])
            state["v_" + key] = np.zeros_like(params[key])
    elif optimizer == 'adagrad':
        for key in params:
            state[key] = np.zeros_like(params[key])
    
    # Prepare dictionaries to store trajectories for each scalar parameter and the loss.
    traj = {}
    for key in params:
        shape = params[key].shape
        for index in np.ndindex(shape):
            traj[f"{key}{index}"] = []
    loss_history = []
    
    n_samples = X_data.shape[0]
    t = 0  # iteration counter (for Adam bias correction)
    
    # Training loop (SGD style: one sample per iteration)
    for epoch in range(epochs):
        indices = np.random.permutation(n_samples)
        for i in indices:
            t += 1
            # Get one sample
            x = X_data[i].reshape(1, in_dim)
            y = Y_class[i].reshape(1, out_dim) if problem_type == 'classification' else Y_reg[i].reshape(1, out_dim)
            
            # --- Forward pass ---
            # Hidden Layer 1
            z1 = np.dot(x, params['W1']) + params['b1']    # shape (1,2)
            a1 = tanh(z1)                                  # shape (1,2)
            # Hidden Layer 2
            z2 = np.dot(a1, params['W2']) + params['b2']     # shape (1,3)
            a2 = tanh(z2)                                  # shape (1,3)
            # Output Layer
            z3 = np.dot(a2, params['W3']) + params['b3']     # shape (1,1)
            if problem_type == 'classification':
                a3 = sigmoid(z3)
                loss = - ( y * np.log(a3 + 1e-8) + (1 - y) * np.log(1 - a3 + 1e-8) )
            else:
                a3 = z3  # linear activation for regression
                loss = 0.5 * (a3 - y)**2
            loss = loss.flatten()[0]
            loss_history.append(loss)
            
            # --- Backpropagation ---
            d3 = a3 - y  # works for both loss functions in this setup (shape: (1,1))
            dW3 = np.dot(a2.T, d3)  # shape (3,1)
            db3 = d3              # shape (1,1)
            
            da2 = np.dot(d3, params['W3'].T)  # shape (1,3)
            dz2 = da2 * tanh_deriv(a2)        # shape (1,3)
            dW2 = np.dot(a1.T, dz2)           # shape (2,3)
            db2 = dz2                       # shape (1,3)
            
            da1 = np.dot(dz2, params['W2'].T) # shape (1,2)
            dz1 = da1 * tanh_deriv(a1)        # shape (1,2)
            dW1 = np.dot(x.T, dz1)            # shape (1,2)
            db1 = dz1                       # shape (1,2)
            
            grads = {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2, 'W3': dW3, 'b3': db3}
            
            # --- Parameter update using chosen optimizer ---
            for key in params:
                grad = grads[key]
                if optimizer == 'sgd':
                    params[key] = params[key] - lr * grad
                elif optimizer == 'momentum':
                    state[key] = 0.9 * state[key] + lr * grad
                    params[key] = params[key] - state[key]
                elif optimizer == 'rmsprop':
                    state[key] = 0.9 * state[key] + (1 - 0.9) * (grad**2)
                    params[key] = params[key] - lr * grad / (np.sqrt(state[key]) + 1e-8)
                elif optimizer == 'adam':
                    state["m_" + key] = 0.9 * state["m_" + key] + (1 - 0.9) * grad
                    state["v_" + key] = 0.999 * state["v_" + key] + (1 - 0.999) * (grad**2)
                    m_hat = state["m_" + key] / (1 - 0.9**t)
                    v_hat = state["v_" + key] / (1 - 0.999**t)
                    params[key] = params[key] - lr * m_hat / (np.sqrt(v_hat) + 1e-8)
                elif optimizer == 'adagrad':
                    state[key] = state[key] + grad**2
                    params[key] = params[key] - lr * grad / (np.sqrt(state[key]) + 1e-8)
                else:
                    raise ValueError("Unknown optimizer.")
            
            # --- Record current parameters ---
            for key in params:
                for index in np.ndindex(params[key].shape):
                    traj[f"{key}{index}"].append(params[key][index])
                    
    return traj, loss_history, params


In [40]:
# %% [code] Train the network for both classification and regression using various optimizers.
optimizers = ['sgd', 'momentum', 'rmsprop', 'adam', 'adagrad']
results = {'classification': {}, 'regression': {}}
epochs = 50
lr = 0.01

for prob in ['classification', 'regression']:
    for opt in optimizers:
        print(f"Training {prob} problem using {opt} optimizer...")
        traj, loss_history, final_params = train_network(problem_type=prob, optimizer=opt, epochs=epochs, lr=lr)
        results[prob][opt] = {'traj': traj, 'loss': loss_history, 'final_params': final_params}
        print("Final loss: {:.5f}".format(loss_history[-1]))


Training classification problem using sgd optimizer...
Final loss: 0.69559
Training classification problem using momentum optimizer...
Final loss: 0.00486
Training classification problem using rmsprop optimizer...
Final loss: 0.00000
Training classification problem using adam optimizer...
Final loss: 0.00037
Training classification problem using adagrad optimizer...
Final loss: 0.24552
Training regression problem using sgd optimizer...
Final loss: 0.00001
Training regression problem using momentum optimizer...
Final loss: 0.04865
Training regression problem using rmsprop optimizer...
Final loss: 0.05732
Training regression problem using adam optimizer...
Final loss: 0.05798
Training regression problem using adagrad optimizer...
Final loss: 0.16503


In [42]:


preset_pairs = {
    "Layer1: W1(0, 0) & b1(0, 0)": ("W1(0, 0)", "b1(0, 0)"),
    "Layer1: W1(0, 1) & b1(0, 1)": ("W1(0, 1)", "b1(0, 1)"),
    
    "Layer2: W2(0, 0) & b2(0, 0)": ("W2(0, 0)", "b2(0, 0)"),
    "Layer2: W2(0, 1) & b2(0, 1)": ("W2(0, 1)", "b2(0, 1)"),
    "Layer2: W2(0, 2) & b2(0, 2)": ("W2(0, 2)", "b2(0, 2)"),
    "Layer2: W2(1, 0) & b2(0, 0)": ("W2(1, 0)", "b2(0, 0)"),
    "Layer2: W2(1, 1) & b2(0, 1)": ("W2(1, 1)", "b2(0, 1)"),
    "Layer2: W2(1, 2) & b2(0, 2)": ("W2(1, 2)", "b2(0, 2)"),
    
    "Output: W3(0, 0) & b3(0, 0)": ("W3(0, 0)", "b3(0, 0)"),
    "Output: W3(1, 0) & b3(0, 0)": ("W3(1, 0)", "b3(0, 0)"),
    "Output: W3(2, 0) & b3(0, 0)": ("W3(2, 0)", "b3(0, 0)")
}

def plot_3d_curve_for_pair(pair_name, problem, optimizer):
    traj = results[problem][optimizer]['traj']
    loss_history = results[problem][optimizer]['loss']
    weight_key, bias_key = preset_pairs[pair_name]
    
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    x_vals = traj[weight_key]
    y_vals = traj[bias_key]
    z_vals = loss_history
    ax.plot(x_vals, y_vals, z_vals, marker='o', label=pair_name)
    ax.set_xlabel("Weight")
    ax.set_ylabel("Bias")
    ax.set_zlabel("Loss")
    ax.set_title(f"3D Evolution: {pair_name}")
    ax.legend()
    plt.show()

def interactive_3d_plot():
    problem_dropdown = Dropdown(options=['classification', 'regression'], description='Problem:')
    optimizer_dropdown = Dropdown(options=optimizers, description='Optimizer:')
    pair_dropdown = Dropdown(options=list(preset_pairs.keys()), description='Param Pair:')
    interact(lambda problem, optimizer, pair: plot_3d_curve_for_pair(pair, problem, optimizer),
             problem=problem_dropdown, optimizer=optimizer_dropdown, pair=pair_dropdown)

print("Interactive 3D visualization for parameter pair evolution (Weight, Bias, Loss):")
interactive_3d_plot()


Interactive 3D visualization for parameter pair evolution (Weight, Bias, Loss):


interactive(children=(Dropdown(description='Problem:', options=('classification', 'regression'), value='classi…

In [46]:
# %% [code] Interactive visualization: Plot parameter trajectory and loss curve.
def plot_parameter(param_name, traj, loss_history):
    """
    Plots the evolution of a chosen parameter and the corresponding loss curve.
    
    Parameters:
        param_name (str): The key for the parameter (e.g., "W1(0, 0)").
        traj (dict): Dictionary containing the trajectory of each scalar parameter.
        loss_history (list): List containing the loss value at each iteration.
    """
    fig, axs = plt.subplots(2, 1, figsize=(10, 8))
    iterations = range(len(traj[param_name]))
    
    # Plot parameter value evolution.
    axs[0].plot(iterations, traj[param_name], label=f"{param_name} trajectory", color="blue")
    axs[0].set_xlabel("Iteration")
    axs[0].set_ylabel("Parameter Value")
    axs[0].set_title(f"Evolution of {param_name}")
    axs[0].legend()
    
    # Plot loss curve.
    axs[1].plot(iterations, loss_history, label="Loss", color="red")
    axs[1].set_xlabel("Iteration")
    axs[1].set_ylabel("Loss")
    axs[1].set_title("Loss Curve")
    axs[1].legend()
    
    plt.tight_layout()
    plt.show()

def interactive_plot():
    # Create dropdowns for problem type and optimizer.
    problem_dropdown = Dropdown(options=['classification', 'regression'], description='Problem:')
    optimizer_dropdown = Dropdown(options=optimizers, description='Optimizer:')
    
    # Update function to load the trajectories and loss curve for the selected configuration.
    def update_plot(problem, optimizer):
        traj = results[problem][optimizer]['traj']
        loss_history = results[problem][optimizer]['loss']
        # Get a sorted list of parameter keys.
        param_keys = sorted(traj.keys())
        param_dropdown = Dropdown(options=param_keys, description='Parameter:')
        interact(lambda param: plot_parameter(param, traj, loss_history), param=param_dropdown)
        
    # Create interactive widget to select problem and optimizer.
    interact(update_plot, problem=problem_dropdown, optimizer=optimizer_dropdown)

print("Interactive visualization for parameter trajectory and loss curve:")
interactive_plot()


Interactive visualization for parameter trajectory and loss curve:


interactive(children=(Dropdown(description='Problem:', options=('classification', 'regression'), value='classi…

In [44]:
# %% [code] (Optional) Display final parameters in a table for a selected configuration.
def display_final_params(problem, optimizer):
    final_params = results[problem][optimizer]['final_params']
    rows = []
    for key in final_params:
        arr = final_params[key]
        for index in np.ndindex(arr.shape):
            rows.append({"Parameter": f"{key}{index}", "Value": arr[index]})
    df = pd.DataFrame(rows)
    display(df)

print("Final parameters for classification problem using Adam optimizer:")
display_final_params('classification', 'adam')


Final parameters for classification problem using Adam optimizer:


,Parameter,Value
0,"W1(0, 0)",2.477521
1,"W1(0, 1)",2.473016
2,"b1(0, 0)",-0.000549
3,"b1(0, 1)",-0.010163
4,"W2(0, 0)",-2.130506
5,"W2(0, 1)",2.088897
6,"W2(0, 2)",1.965438
7,"W2(1, 0)",-2.381132
8,"W2(1, 1)",2.475332
9,"W2(1, 2)",2.314879


In [11]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_deriv(x):
    s = sigmoid(x)
    return s * (1 - s)

def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


# Input: Single feature
X = np.array([[0], [1], [2], [3]])      # shape: (4, 1)
y = np.array([[0], [0], [1], [1]])      # binary classes


np.random.seed(0)

# Weights initialization
W1 = np.random.randn(1, 2)    # Input → H1
b1 = np.zeros((1, 2))

W2 = np.random.randn(2, 3)    # H1 → H2
b2 = np.zeros((1, 3))

W3 = np.random.randn(3, 1)    # H2 → Output
b3 = np.zeros((1, 1))

lr = 0.1
epochs = 1000

for epoch in range(epochs):
    for i in range(len(X)):
        x_i = X[i].reshape(1, 1)
        y_i = y[i].reshape(1, 1)

        # Forward pass
        z1 = np.dot(x_i, W1) + b1
        a1 = sigmoid(z1)

        z2 = np.dot(a1, W2) + b2
        a2 = sigmoid(z2)

        z3 = np.dot(a2, W3) + b3
        a3 = sigmoid(z3)  # final output

        # Backpropagation
        error = a3 - y_i
        dz3 = error * sigmoid_deriv(z3)
        dW3 = np.dot(a2.T, dz3)
        db3 = dz3

        dz2 = np.dot(dz3, W3.T) * sigmoid_deriv(z2)
        dW2 = np.dot(a1.T, dz2)
        db2 = dz2

        dz1 = np.dot(dz2, W2.T) * sigmoid_deriv(z1)
        dW1 = np.dot(x_i.T, dz1)
        db1 = dz1

        # SGD weight updates
        W3 -= lr * dW3
        b3 -= lr * db3
        W2 -= lr * dW2
        b2 -= lr * db2
        W1 -= lr * dW1
        b1 -= lr * db1

    # Print loss every 200 epochs
    if epoch % 200 == 0:
        z1 = np.dot(X, W1) + b1
        a1 = sigmoid(z1)
        z2 = np.dot(a1, W2) + b2
        a2 = sigmoid(z2)
        z3 = np.dot(a2, W3) + b3
        a3 = sigmoid(z3)
        print(f"Epoch {epoch}, Loss:", mse_loss(y, a3))


# Regression: y = 2x + 1
X = np.array([[0], [1], [2], [3]])
y = np.array([[1], [3], [5], [7]])


np.random.seed(0)

W1 = np.random.randn(1, 2)
b1 = np.zeros((1, 2))

W2 = np.random.randn(2, 3)
b2 = np.zeros((1, 3))

W3 = np.random.randn(3, 1)
b3 = np.zeros((1, 1))

lr = 0.01
epochs = 1000

for epoch in range(epochs):
    for i in range(len(X)):
        x_i = X[i].reshape(1, 1)
        y_i = y[i].reshape(1, 1)

        # Forward
        z1 = np.dot(x_i, W1) + b1
        a1 = sigmoid(z1)

        z2 = np.dot(a1, W2) + b2
        a2 = sigmoid(z2)

        z3 = np.dot(a2, W3) + b3
        a3 = z3  # linear output

        # Backprop
        error = a3 - y_i
        dz3 = error
        dW3 = np.dot(a2.T, dz3)
        db3 = dz3

        dz2 = np.dot(dz3, W3.T) * sigmoid_deriv(z2)
        dW2 = np.dot(a1.T, dz2)
        db2 = dz2

        dz1 = np.dot(dz2, W2.T) * sigmoid_deriv(z1)
        dW1 = np.dot(x_i.T, dz1)
        db1 = dz1

        # SGD update
        W3 -= lr * dW3
        b3 -= lr * db3
        W2 -= lr * dW2
        b2 -= lr * db2
        W1 -= lr * dW1
        b1 -= lr * db1

    if epoch % 200 == 0:
        z1 = np.dot(X, W1) + b1
        a1 = sigmoid(z1)
        z2 = np.dot(a1, W2) + b2
        a2 = sigmoid(z2)
        a3 = np.dot(a2, W3) + b3
        print(f"Epoch {epoch}, Loss:", mse_loss(y, a3))


Epoch 0, Loss: 0.2558459724308267
Epoch 200, Loss: 0.2430018815932955
Epoch 400, Loss: 0.23274195420503574
Epoch 600, Loss: 0.20407100884170576
Epoch 800, Loss: 0.14152369095082737
Epoch 0, Loss: 14.889492447380668
Epoch 200, Loss: 1.649748429805546
Epoch 400, Loss: 0.2702468661452262
Epoch 600, Loss: 0.14984971501375305
Epoch 800, Loss: 0.11968958658751326
